# Personalized Hearing Enhancement Demo\nThis walkthrough uses existing repository modules for deterministic, lightweight CPU demonstration.

In [ ]:
from pathlib import Path\nimport matplotlib.pyplot as plt\nimport torch\nfrom IPython.display import Audio, display\nfrom omegaconf import OmegaConf\nfrom personalized_hearing_enhancement.data.download import download_datasets\nfrom personalized_hearing_enhancement.evaluation.demo_audio import run_demo_audio\nfrom personalized_hearing_enhancement.models.conditioned_tasnet import ConditionedConvTasNet\nfrom personalized_hearing_enhancement.models.tasnet import ConvTasNet\nfrom personalized_hearing_enhancement.simulation.hearing_loss import apply_hearing_loss\nfrom personalized_hearing_enhancement.utils.audio import load_audio, save_audio\nfrom personalized_hearing_enhancement.utils.plotting import save_spectrogram_plot, save_waveform_plot\nfrom personalized_hearing_enhancement.utils.repro import set_global_seed\ncfg = OmegaConf.load('personalized_hearing_enhancement/configs/default.yaml')\nset_global_seed(int(cfg.seed))\nout_dir = Path('outputs/notebook_demo')\nout_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
download_datasets(cfg.paths.data_cache, selected=['librispeech_dev','musan','rir'])\nsrc = list((Path(cfg.paths.data_cache)/'LibriSpeech'/'dev-clean').rglob('*.flac'))[0]\nclean = load_audio(src, sr=int(cfg.sample_rate))[:int(cfg.sample_rate*3)]\nag = torch.tensor([[20,25,30,45,60,65,70,75]], dtype=torch.float32)\ndegraded = apply_hearing_loss(clean.unsqueeze(0), ag, sr=int(cfg.sample_rate)).squeeze(0)\nsave_audio(out_dir/'clean.wav', clean, int(cfg.sample_rate))\nsave_audio(out_dir/'degraded.wav', degraded, int(cfg.sample_rate))

In [ ]:
save_waveform_plot({'clean': clean, 'degraded': degraded}, out_dir/'waveforms.png', sr=int(cfg.sample_rate))\nsave_spectrogram_plot({'clean': clean, 'degraded': degraded}, out_dir/'spectrograms.png', sr=int(cfg.sample_rate))\nplt.figure(figsize=(6,2))\nplt.bar(['250','500','1k','2k','4k','6k','8k','9k'], ag.squeeze(0).tolist())\nplt.title('Audiogram (dB loss)')\nplt.tight_layout(); plt.show()

In [ ]:
base = ConvTasNet(**dict(cfg.model.tasnet))\ncond = ConditionedConvTasNet(**dict(cfg.model.tasnet))\nb_ckpt = Path('outputs/checkpoints/baseline_best.pt')\nc_ckpt = Path('outputs/checkpoints/conditioned_best.pt')\nif b_ckpt.exists():\n    base.load_state_dict(torch.load(b_ckpt, map_location='cpu')['model'])\nif c_ckpt.exists():\n    cond.load_state_dict(torch.load(c_ckpt, map_location='cpu')['model'])\nwith torch.no_grad():\n    b = base(degraded.unsqueeze(0)).squeeze(0)\n    c = cond(degraded.unsqueeze(0), ag).squeeze(0)\nsave_audio(out_dir/'baseline.wav', b, int(cfg.sample_rate))\nsave_audio(out_dir/'conditioned.wav', c, int(cfg.sample_rate))

In [ ]:
display(Audio((out_dir/'clean.wav').as_posix()))\ndisplay(Audio((out_dir/'degraded.wav').as_posix()))\ndisplay(Audio((out_dir/'baseline.wav').as_posix()))\ndisplay(Audio((out_dir/'conditioned.wav').as_posix()))

In [ ]:
try:\n    run_demo_audio((out_dir/'clean.wav').as_posix(), 'personalized_hearing_enhancement/configs/default.yaml',\n                  'outputs/checkpoints/baseline_best.pt', 'outputs/checkpoints/conditioned_best.pt',\n                  '20,25,30,45,60,65,70,75', 'outputs', 'notebook_demo')\nexcept Exception as e:\n    print('Demo audio fallback warning:', e)\nprint('Notebook outputs written to', out_dir)

Optional video debug step: run CLI command `python -m personalized_hearing_enhancement.cli.main process-video --input sample.mp4 --run-name notebook_demo` if ffmpeg and sample video are available.